# Kapitel 6 - Koduppgift 8: HR Employee Data Klustring

I denna uppgift genomför vi klusteranalys på datasetet 'hr_employee_data.xlsx'. Vi betraktar alla variabler som features (X) och modellerar inte med avseende på 'left' som i kapitel 4.

## Importera bibliotek och ladda data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

# Sätt stil för plots
plt.style.use('default')
sns.set_palette('husl')

print('Bibliotek importerade!')

## Ladda och utforska data

In [ ]:
# Ladda HR employee dataset
df = pd.read_excel('hr_employee_data.xlsx')

print(f'Dataset shape: {df.shape}')
print(f'\nKolumner: {list(df.columns)}')
print(f'\nFörsta 5 rader:')
print(df.head())

In [ ]:
# Grundläggande information
print('Dataset info:')
print(df.info())
print(f'\nSaknade värden:')
print(df.isnull().sum())
print(f'\nBeskrivande statistik:')
print(df.describe())

In [ ]:
# Analysera kategoriska variabler
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print(f'Kategoriska kolumner: {categorical_cols}')
print(f'Numeriska kolumner: {numerical_cols}')

print('\nUnika värden i kategoriska kolumner:')
for col in categorical_cols:
    print(f'\n{col}: {df[col].unique()}')

## Data preprocessing för klustring

In [ ]:
# Kopiera data för säkerhet
df_clean = df.copy()

# Hantera saknade värden
print('Saknade värden före rensning:')
print(df_clean.isnull().sum())

# Ta bort rader med saknade värden
df_clean = df_clean.dropna()

print(f'\nData efter rensning: {df_clean.shape}')
print('Saknade värden efter rensning:')
print(df_clean.isnull().sum())

In [ ]:
# Förbered features för klustring
# Vi använder alla variabler som features (inte 'left' som target)
feature_cols = [col for col in df_clean.columns if col != 'left']
X = df_clean[feature_cols].copy()

print(f'Features för klustring: {feature_cols}')
print(f'Features shape: {X.shape}')
print(f'\nFörsta 5 rader:')
print(X.head())

In [ ]:
# Encode kategoriska variabler
X_encoded = X.copy()
label_encoders = {}

for col in categorical_cols:
    if col in X_encoded.columns:
        le = LabelEncoder()
        X_encoded[col] = le.fit_transform(X_encoded[col])
        label_encoders[col] = le
        print(f'{col} encoded: {le.classes_}')

print(f'\nData efter encoding: {X_encoded.shape}')
print(f'\nFörsta 5 rader efter encoding:')
print(X_encoded.head())

## Standardisera data

In [ ]:
# Standardisera features (viktigt för klustring)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

print(f'Data efter standardisering: {X_scaled.shape}')
print(f'\nGenomsnitt: {X_scaled.mean(axis=0)[:5]}')
print(f'Standardavvikelse: {X_scaled.std(axis=0)[:5]}')

## Välja optimalt antal kluster

In [ ]:
# Testa olika antal kluster
k_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

print('Inertia och Silhouette scores:')
for k, inertia, sil in zip(k_range, inertias, silhouette_scores):
    print(f'k={k}: Inertia={inertia:.2f}, Silhouette={sil:.3f}')

In [ ]:
# Visualisera Elbow method och Silhouette scores
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Elbow method
axes[0].plot(k_range, inertias, 'bo-')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Antal kluster (k)')
axes[0].set_ylabel('Inertia')
axes[0].grid(True, alpha=0.3)

# Silhouette scores
axes[1].plot(k_range, silhouette_scores, 'ro-')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Antal kluster (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## K-means klustring

In [ ]:
# Välj optimalt antal kluster baserat på Silhouette score
best_k = k_range[np.argmax(silhouette_scores)]
print(f'Optimalt antal kluster baserat på Silhouette score: {best_k}')

# K-means klustring
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# Lägg till klusterlabels till originaldata
df_clustered = df_clean.copy()
df_clustered['Cluster'] = cluster_labels

print(f'\nKlustring slutförd med {best_k} kluster!')
print(f'\nKlusterfördelning:')
print(df_clustered['Cluster'].value_counts().sort_index())

## Analysera kluster

In [ ]:
# Analysera varje kluster
print('Detaljerad klusteranalys:')
print('=' * 50)

for cluster in sorted(df_clustered['Cluster'].unique()):
    cluster_data = df_clustered[df_clustered['Cluster'] == cluster]
    
    print(f'\nKLUSTER {cluster}:')
    print(f'  Antal anställda: {len(cluster_data)}')
    print(f'  Andel av totalt: {len(cluster_data)/len(df_clustered)*100:.1f}%')
    
    # Analysera numeriska variabler
    print(f'  \n  NUMERISKA VARIABLER:')
    for col in numerical_cols:
        if col in cluster_data.columns:
            mean_val = cluster_data[col].mean()
            print(f'    {col}: {mean_val:.2f}')
    
    # Analysera kategoriska variabler
    print(f'  \n  KATEGORISKA VARIABLER:')
    for col in categorical_cols:
        if col in cluster_data.columns:
            most_common = cluster_data[col].mode().iloc[0] if len(cluster_data[col].mode()) > 0 else 'N/A'
            print(f'    {col}: {most_common}')

In [ ]:
# Visualisera kluster med viktiga variabler
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

# Satisfaction level vs Last evaluation
sns.scatterplot(data=df_clustered, x='satisfaction_level', y='last_evaluation', 
                hue='Cluster', palette='husl', ax=axes[0])
axes[0].set_title('Satisfaction Level vs Last Evaluation')
axes[0].grid(True, alpha=0.3)

# Number of projects vs Average monthly hours
sns.scatterplot(data=df_clustered, x='number_project', y='average_montly_hours', 
                hue='Cluster', palette='husl', ax=axes[1])
axes[1].set_title('Number of Projects vs Average Monthly Hours')
axes[1].grid(True, alpha=0.3)

# Time spend company vs Salary
sns.boxplot(data=df_clustered, x='Cluster', y='time_spend_company', 
            palette='husl', ax=axes[2])
axes[2].set_title('Time Spend Company by Cluster')
axes[2].grid(True, alpha=0.3)

# Salary distribution by cluster
salary_counts = df_clustered.groupby(['Cluster', 'salary']).size().unstack(fill_value=0)
salary_counts.plot(kind='bar', stacked=True, ax=axes[3], color=['lightblue', 'orange', 'lightgreen'])
axes[3].set_title('Salary Distribution by Cluster')
axes[3].set_xlabel('Cluster')
axes[3].set_ylabel('Count')
axes[3].legend(title='Salary')
axes[3].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## Hierarkisk klustring för jämförelse

In [ ]:
# Hierarkisk klustring
agg_clustering = AgglomerativeClustering(n_clusters=best_k)
agg_labels = agg_clustering.fit_predict(X_scaled)

# Lägg till hierarkiska klusterlabels
df_clustered['Cluster_Hierarchical'] = agg_labels

print(f'Hierarkisk klustring slutförd med {best_k} kluster!')
print(f'\nHierarkisk klusterfördelning:')
print(df_clustered['Cluster_Hierarchical'].value_counts().sort_index())

# Jämför Silhouette scores
kmeans_sil = silhouette_score(X_scaled, cluster_labels)
agg_sil = silhouette_score(X_scaled, agg_labels)

print(f'\nSilhouette scores:')
print(f'K-means: {kmeans_sil:.3f}')
print(f'Hierarkisk: {agg_sil:.3f}')

In [ ]:
# Skapa dendrogram
linkage_matrix = linkage(X_scaled, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(linkage_matrix, truncate_mode='level', p=5)
plt.title('Dendrogram för Hierarkisk Klustring')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.grid(True, alpha=0.3)
plt.show()

## Praktiska tillämpningar för HR

In [ ]:
# Analysera vilka anställda som lämnade företaget per kluster
print('Analys av anställda som lämnade företaget:')
print('=' * 50)

for cluster in sorted(df_clustered['Cluster'].unique()):
    cluster_data = df_clustered[df_clustered['Cluster'] == cluster]
    left_rate = cluster_data['left'].mean()
    
    print(f'\nKluster {cluster}:')
    print(f'  Antal anställda: {len(cluster_data)}')
    print(f'  Andel som lämnade: {left_rate:.1%}')
    print(f'  Antal som lämnade: {cluster_data["left"].sum()}')

In [ ]:
# Rekommendationer baserat på klusteranalys
print('HR REKOMMENDATIONER BASERAT PÅ KLUSTERANALYS:')
print('=' * 60)

print('\n1. KLUSTER-SPECIFIKA STRATEGIER:')
print('   - Olika retention-strategier för olika kluster')
print('   - Anpassade utvecklingsprogram')
print('   - Olika belöningssystem')

print('\n2. PROAKTIVA ÅTGÄRDER:')
print('   - Identifiera riskgrupper tidigt')
print('   - Anpassade interventionsprogram')
print('   - Kontinuerlig övervakning av kluster')

print('\n3. REKRYTERINGSSTRATEGI:')
print('   - Fokusera på kandidater som passar stabila kluster')
print('   - Undvik kandidater som liknar riskgrupper')
print('   - Anpassade onboarding-processer')

print('\n4. ORGANISATIONSUTVECKLING:')
print('   - Analysera vad som gör vissa kluster framgångsrika')
print('   - Replikera framgångsrika mönster')
print('   - Förbättra arbetsmiljö baserat på klusterbehov')

## Detaljerad klusterprofilering

In [ ]:
# Skapa detaljerade profiler för varje kluster
print('DETALJERADE KLUSTERPROFILER:')
print('=' * 40)

for cluster in sorted(df_clustered['Cluster'].unique()):
    cluster_data = df_clustered[df_clustered['Cluster'] == cluster]
    
    print(f'\n\nKLUSTER {cluster} - "{get_cluster_name(cluster, cluster_data)}"')
    print('-' * 50)
    
    # Profilering baserat på data
    avg_satisfaction = cluster_data['satisfaction_level'].mean()
    avg_evaluation = cluster_data['last_evaluation'].mean()
    avg_projects = cluster_data['number_project'].mean()
    avg_hours = cluster_data['average_montly_hours'].mean()
    avg_time_company = cluster_data['time_spend_company'].mean()
    left_rate = cluster_data['left'].mean()
    
    print(f'PROFIL:')
    satisfaction_level = "Hög" if avg_satisfaction > 0.6 else "Låg"
    evaluation_level = "Hög" if avg_evaluation > 0.7 else "Låg"
    projects_level = "Många" if avg_projects > 4 else "Få"
    hours_level = "Hög" if avg_hours > 200 else "Normal"
    
    print(f'  • Satisfaction Level: {avg_satisfaction:.2f} ({satisfaction_level})')
    print(f'  • Last Evaluation: {avg_evaluation:.2f} ({evaluation_level})')
    print(f'  • Antal Projekt: {avg_projects:.1f} ({projects_level})')
    print(f'  • Månadsarbetstid: {avg_hours:.0f}h ({hours_level})')
    print(f'  • Tid på företag: {avg_time_company:.1f} år')
    print(f'  • Risk att lämna: {left_rate:.1%}')
    
    # Rekommendationer
    print(f'\nREKOMMENDATIONER:')
    if left_rate > 0.3:
        print(f'    HÖG RISK - Kräver omedelbar uppmärksamhet')
        if avg_satisfaction < 0.5:
            print(f'     • Förbättra arbetsmiljö och kommunikation')
        if avg_hours > 250:
            print(f'     • Minska arbetsbelastning')
        if avg_projects > 5:
            print(f'     • Prioritera projekt och minska antal')
    elif left_rate > 0.1:
        print(f'   MEDELRISK - Proaktiva åtgärder rekommenderas')
        print(f'     • Regelbunden feedback och utvecklingssamtal')
        print(f'     • Karriärutvecklingsmöjligheter')
    else:
        print(f'   LÅG RISK - Behåll och utveckla')
        print(f'     • Använd som mentors för nya anställda')
        print(f'     • Erbjud avancerade utvecklingsmöjligheter')

def get_cluster_name(cluster, cluster_data):
    """Ge ett beskrivande namn till klustret baserat på dess egenskaper"""
    avg_satisfaction = cluster_data['satisfaction_level'].mean()
    avg_evaluation = cluster_data['last_evaluation'].mean()
    left_rate = cluster_data['left'].mean()
    
    if left_rate > 0.3:
        if avg_satisfaction < 0.4:
            return "Missnöjda Anställda"
        else:
            return "Riskgrupp"
    elif avg_satisfaction > 0.7 and avg_evaluation > 0.7:
        return "Stjärnanställda"
    elif avg_satisfaction > 0.6:
        return "Nöjda Anställda"
    else:
        return "Genomsnittliga Anställda"

## Sammanfattning

**Vad vi har gjort:**

1. **Laddat HR employee dataset** och utforskat dess struktur
2. **Preprocessat data** genom encoding av kategoriska variabler och standardisering
3. **Valt optimalt antal kluster** baserat på Silhouette score
4. **Applicerat K-means och Hierarkisk klustring** för att segmentera anställda
5. **Analyserat kluster** för att förstå deras egenskaper och risknivåer
6. **Utvecklat praktiska rekommendationer** för HR-strategier

**Viktiga insikter:**
- **Segmentering**: Anställda kan delas in i meningsfulla grupper baserat på deras egenskaper
- **Riskidentifiering**: Vissa kluster har högre risk att lämna företaget
- **Personaliserade strategier**: Olika kluster kräver olika HR-åtgärder
- **Proaktiv hantering**: Klustring hjälper att identifiera problem innan de blir kritiska

**Praktiska tillämpningar:**
- **Retention-strategier**: Anpassade program för att behålla värdefulla anställda
- **Rekrytering**: Fokusera på kandidater som passar stabila kluster
- **Utvecklingsprogram**: Anpassade utbildningar baserat på klusterbehov
- **Belöningssystem**: Olika incitament för olika anställdagrupper
- **Arbetsmiljö**: Förbättringar baserat på klusterpreferenser